# 02 Weather Data Exploration

This notebook tests whether weather data can be retrieved for the temperature-based Polymarket project.

The goal is not to build the final forecasting model yet. The goal is to check whether we can obtain:

1. realised or proxy-realised 2m temperature data;
2. historical forecast data;
3. lead-time-specific forecast data;
4. daily maximum temperature for locations linked to Polymarket contracts.

This is a feasibility notebook for Priority 6.

In [2]:
import requests
import pandas as pd
import json
import os
from datetime import datetime, timezone
from pprint import pprint
import re

ARCHIVE_BASE = "https://archive-api.open-meteo.com/v1/archive"
HIST_FORECAST_BASE = "https://historical-forecast-api.open-meteo.com/v1/forecast"
PREVIOUS_RUNS_BASE = "https://previous-runs-api.open-meteo.com/v1/forecast"

def get_json(url, params=None, timeout=30):
    response = requests.get(url, params=params, timeout=timeout)
    print("URL:", response.url)
    print("Status code:", response.status_code)
    response.raise_for_status()
    return response.json()

def pretty(obj, max_chars=3000):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated, total length = {len(text)} characters")

## 1. Candidate locations

These coordinates are not yet final settlement-source matches. They are practical first approximations:

- Hong Kong Observatory area for Hong Kong contracts;
- London City Airport for London contracts;
- LaGuardia Airport for New York City contracts.

The key dissertation issue is that Polymarket contracts settle against specific sources/stations, so later we must match the realised source exactly where possible.

In [4]:
locations = pd.DataFrame([
    {
        "city": "Hong Kong",
        "contract_reference": "Hong Kong Observatory Daily Extract",
        "latitude": 22.3022,
        "longitude": 114.1746,
        "timezone": "Asia/Hong_Kong",
        "polymarket_unit": "Celsius",
        "notes": "Approximate coordinate near Hong Kong Observatory. Settlement source still needs exact HKO matching."
    },
    {
        "city": "London",
        "contract_reference": "London City Airport / EGLC",
        "latitude": 51.5053,
        "longitude": 0.0553,
        "timezone": "Europe/London",
        "polymarket_unit": "Celsius",
        "notes": "Approximate coordinate for London City Airport."
    },
    {
        "city": "New York City",
        "contract_reference": "LaGuardia Airport / KLGA",
        "latitude": 40.7769,
        "longitude": -73.8740,
        "timezone": "America/New_York",
        "polymarket_unit": "Fahrenheit",
        "notes": "Approximate coordinate for LaGuardia Airport. Open-Meteo can return Fahrenheit directly."
    },
])

locations

,city,contract_reference,latitude,longitude,timezone,polymarket_unit,notes
0,Hong Kong,Hong Kong Observatory Daily Extract,22.3022,114.1746,Asia/Hong_Kong,Celsius,Approximate coordinate near Hong Kong Observat...
1,London,London City Airport / EGLC,51.5053,0.0553,Europe/London,Celsius,Approximate coordinate for London City Airport.
2,New York City,LaGuardia Airport / KLGA,40.7769,-73.8740,America/New_York,Fahrenheit,Approximate coordinate for LaGuardia Airport. ...


## 2. Weather-source feasibility matrix

This records which sources are immediately usable and which require further work.

In [6]:
source_audit = pd.DataFrame([
    {
        "source": "Open-Meteo Historical Weather API",
        "type": "historical/reanalysis proxy",
        "api_key_required": "No",
        "tested_in_this_notebook": "Yes",
        "strength": "Easy access, daily and hourly temperature, good for first baseline.",
        "limitation": "Not necessarily the exact Polymarket settlement source."
    },
    {
        "source": "Open-Meteo Previous Runs API",
        "type": "lead-time forecast data",
        "api_key_required": "No",
        "tested_in_this_notebook": "Yes",
        "strength": "Gives fixed lead-time forecast variables such as previous_day1 to previous_day7.",
        "limitation": "Still coordinate/grid based, not exact station settlement."
    },
    {
        "source": "Open-Meteo Historical Forecast API",
        "type": "historical forecast time series",
        "api_key_required": "No",
        "tested_in_this_notebook": "Optional",
        "strength": "Useful for bias correction and forecast-model comparison.",
        "limitation": "Continuous stitched forecast series is not the same as one specific model run."
    },
    {
        "source": "Hong Kong Observatory Daily Extract",
        "type": "settlement realised data",
        "api_key_required": "No / web data",
        "tested_in_this_notebook": "No",
        "strength": "Exact settlement source for Hong Kong contracts.",
        "limitation": "Need robust download/parsing method."
    },
    {
        "source": "Wunderground station pages",
        "type": "settlement realised data for some markets",
        "api_key_required": "Unclear",
        "tested_in_this_notebook": "No",
        "strength": "Exact source/station for London and NYC contracts if accessible.",
        "limitation": "May be difficult to automate cleanly."
    },
    {
        "source": "ECMWF / Met Office / Windy",
        "type": "forecast data",
        "api_key_required": "Depends on source",
        "tested_in_this_notebook": "No",
        "strength": "Potentially closer to professional weather forecasts.",
        "limitation": "Access, licensing, and API constraints need checking."
    },
])

source_audit

,source,type,api_key_required,tested_in_this_notebook,strength,limitation
0,Open-Meteo Historical Weather API,historical/reanalysis proxy,No,Yes,"Easy access, daily and hourly temperature, goo...",Not necessarily the exact Polymarket settlemen...
1,Open-Meteo Previous Runs API,lead-time forecast data,No,Yes,Gives fixed lead-time forecast variables such ...,"Still coordinate/grid based, not exact station..."
2,Open-Meteo Historical Forecast API,historical forecast time series,No,Optional,Useful for bias correction and forecast-model ...,Continuous stitched forecast series is not the...
3,Hong Kong Observatory Daily Extract,settlement realised data,No / web data,No,Exact settlement source for Hong Kong contracts.,Need robust download/parsing method.
4,Wunderground station pages,settlement realised data for some markets,Unclear,No,Exact source/station for London and NYC contra...,May be difficult to automate cleanly.
5,ECMWF / Met Office / Windy,forecast data,Depends on source,No,Potentially closer to professional weather for...,"Access, licensing, and API constraints need ch..."


## 3. Retrieve historical/reanalysis daily maximum temperature

This first test uses Open-Meteo's Historical Weather API as a realised-temperature proxy.

For Hong Kong, the target date is chosen to match the sample Polymarket event used in the API notebook.

In [8]:
event_date = "2026-06-10"

archive_results = []

for _, loc in locations.iterrows():
    temp_unit = "fahrenheit" if loc["polymarket_unit"] == "Fahrenheit" else "celsius"
    
    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "start_date": event_date,
        "end_date": event_date,
        "daily": "temperature_2m_max",
        "hourly": "temperature_2m",
        "timezone": loc["timezone"],
        "temperature_unit": temp_unit,
    }
    
    try:
        data = get_json(ARCHIVE_BASE, params=params)
        daily = data.get("daily", {})
        
        archive_results.append({
            "city": loc["city"],
            "date": daily.get("time", [None])[0] if daily.get("time") else None,
            "temperature_2m_max": daily.get("temperature_2m_max", [None])[0] if daily.get("temperature_2m_max") else None,
            "temperature_unit": data.get("daily_units", {}).get("temperature_2m_max"),
            "timezone": data.get("timezone"),
            "latitude_returned": data.get("latitude"),
            "longitude_returned": data.get("longitude"),
            "source": "Open-Meteo Historical Weather API",
            "status": "success",
        })
    except Exception as e:
        archive_results.append({
            "city": loc["city"],
            "date": event_date,
            "temperature_2m_max": None,
            "temperature_unit": temp_unit,
            "timezone": loc["timezone"],
            "latitude_returned": None,
            "longitude_returned": None,
            "source": "Open-Meteo Historical Weather API",
            "status": f"error: {e}",
        })

archive_daily_df = pd.DataFrame(archive_results)
archive_daily_df

URL: https://archive-api.open-meteo.com/v1/archive?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&daily=temperature_2m_max&hourly=temperature_2m&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
URL: https://archive-api.open-meteo.com/v1/archive?latitude=51.5053&longitude=0.0553&start_date=2026-06-10&end_date=2026-06-10&daily=temperature_2m_max&hourly=temperature_2m&timezone=Europe%2FLondon&temperature_unit=celsius
Status code: 200
URL: https://archive-api.open-meteo.com/v1/archive?latitude=40.7769&longitude=-73.874&start_date=2026-06-10&end_date=2026-06-10&daily=temperature_2m_max&hourly=temperature_2m&timezone=America%2FNew_York&temperature_unit=fahrenheit
Status code: 200


,city,date,temperature_2m_max,temperature_unit,timezone,latitude_returned,longitude_returned,source,status
0,Hong Kong,2026-06-10,27.4,°C,Asia/Hong_Kong,22.319859,114.198555,Open-Meteo Historical Weather API,success
1,London,2026-06-10,17.4,°C,Europe/London,51.493847,0.000000,Open-Meteo Historical Weather API,success
2,New York City,2026-06-10,83.3,°F,America/New_York,40.808434,-73.892060,Open-Meteo Historical Weather API,success


## 4. Inspect hourly temperature for Hong Kong

Daily max is useful, but hourly data is useful for checking the construction of the daily maximum.

In [10]:
hk = locations[locations["city"] == "Hong Kong"].iloc[0]

hk_archive = get_json(
    ARCHIVE_BASE,
    params={
        "latitude": hk["latitude"],
        "longitude": hk["longitude"],
        "start_date": event_date,
        "end_date": event_date,
        "hourly": "temperature_2m",
        "daily": "temperature_2m_max",
        "timezone": hk["timezone"],
        "temperature_unit": "celsius",
    }
)

pretty(hk_archive, max_chars=2000)

URL: https://archive-api.open-meteo.com/v1/archive?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&hourly=temperature_2m&daily=temperature_2m_max&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
{
  "latitude": 22.319859,
  "longitude": 114.198555,
  "generationtime_ms": 0.05841255187988281,
  "utc_offset_seconds": 28800,
  "timezone": "Asia/Hong_Kong",
  "timezone_abbreviation": "GMT+8",
  "elevation": 38.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "°C"
  },
  "hourly": {
    "time": [
      "2026-06-10T00:00",
      "2026-06-10T01:00",
      "2026-06-10T02:00",
      "2026-06-10T03:00",
      "2026-06-10T04:00",
      "2026-06-10T05:00",
      "2026-06-10T06:00",
      "2026-06-10T07:00",
      "2026-06-10T08:00",
      "2026-06-10T09:00",
      "2026-06-10T10:00",
      "2026-06-10T11:00",
      "2026-06-10T12:00",
      "2026-06-10T13:00",
      "2026-06-10T14:00",
      "2026-06-10T15:00",
      "2026-06-10T1

In [11]:
hk_hourly_df = pd.DataFrame({
    "time": hk_archive["hourly"]["time"],
    "temperature_2m": hk_archive["hourly"]["temperature_2m"],
})

hk_hourly_df["time"] = pd.to_datetime(hk_hourly_df["time"])
hk_hourly_df["date"] = hk_hourly_df["time"].dt.date

display(hk_hourly_df.head())
display(hk_hourly_df.tail())

print("Hourly max temperature:", hk_hourly_df["temperature_2m"].max())
print("Daily API max temperature:", hk_archive["daily"]["temperature_2m_max"][0])

,time,temperature_2m,date
0,2026-06-10 00:00:00,23.1,2026-06-10
1,2026-06-10 01:00:00,22.9,2026-06-10
2,2026-06-10 02:00:00,23.5,2026-06-10
3,2026-06-10 03:00:00,23.2,2026-06-10
4,2026-06-10 04:00:00,23.0,2026-06-10


,time,temperature_2m,date
19,2026-06-10 19:00:00,24.5,2026-06-10
20,2026-06-10 20:00:00,25.0,2026-06-10
21,2026-06-10 21:00:00,24.5,2026-06-10
22,2026-06-10 22:00:00,24.5,2026-06-10
23,2026-06-10 23:00:00,24.5,2026-06-10


Hourly max temperature: 27.4
Daily API max temperature: 27.4


## 5. Retrieve lead-time forecast data using Previous Runs API

This is important for the dissertation because Polymarket prices evolve before settlement. We need forecasts issued before the event date, not only realised values after the event.

The Previous Runs API gives lead-time-specific forecast variables such as:

* `temperature_2m`: day-0 forecast / current run;
* `temperature_2m_previous_day1`: forecast made around one day before the valid date;
* `temperature_2m_previous_day2`: forecast made around two days before the valid date;
* `temperature_2m_previous_day3`: forecast made around three days before the valid date;
* `temperature_2m_previous_day4`: forecast made around four days before the valid date;
* `temperature_2m_previous_day5`: forecast made around five days before the valid date;
* `temperature_2m_previous_day6`: forecast made around six days before the valid date;
* `temperature_2m_previous_day7`: forecast made around seven days before the valid date.

In this notebook, we retrieve hourly temperature forecasts for day-0 to day-7 lead times and then compute the daily maximum temperature for each forecast lead time.

This gives the first version of the project’s weather-forecast dataset:

`city + event date + forecast lead time + forecast daily max temperature`

This structure can later be compared with realised settlement temperatures and Polymarket-implied probabilities.


In [13]:
previous_run_vars = ["temperature_2m"] + [
    f"temperature_2m_previous_day{i}" for i in range(1, 8)
]

previous_runs_params = {
    "latitude": hk["latitude"],
    "longitude": hk["longitude"],
    "start_date": event_date,
    "end_date": event_date,
    "hourly": ",".join(previous_run_vars),
    "timezone": hk["timezone"],
    "temperature_unit": "celsius",
}

try:
    hk_prev_runs = get_json(PREVIOUS_RUNS_BASE, params=previous_runs_params)
    pretty(hk_prev_runs, max_chars=2500)
except Exception as e:
    hk_prev_runs = None
    print("Previous Runs API failed:", e)

URL: https://previous-runs-api.open-meteo.com/v1/forecast?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&hourly=temperature_2m%2Ctemperature_2m_previous_day1%2Ctemperature_2m_previous_day2%2Ctemperature_2m_previous_day3%2Ctemperature_2m_previous_day4%2Ctemperature_2m_previous_day5%2Ctemperature_2m_previous_day6%2Ctemperature_2m_previous_day7&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
{
  "latitude": 22.319859,
  "longitude": 114.198555,
  "generationtime_ms": 14.791727066040039,
  "utc_offset_seconds": 28800,
  "timezone": "Asia/Hong_Kong",
  "timezone_abbreviation": "GMT+8",
  "elevation": 38.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "°C",
    "temperature_2m_previous_day1": "°C",
    "temperature_2m_previous_day2": "°C",
    "temperature_2m_previous_day3": "°C",
    "temperature_2m_previous_day4": "°C",
    "temperature_2m_previous_day5": "°C",
    "temperature_2m_previous_day6": "°C",
    "temperature_

In [14]:
if hk_prev_runs is not None and "hourly" in hk_prev_runs:
    hk_prev_hourly_df = pd.DataFrame(hk_prev_runs["hourly"])
    hk_prev_hourly_df["time"] = pd.to_datetime(hk_prev_hourly_df["time"])
    
    temp_cols = [c for c in hk_prev_hourly_df.columns if c.startswith("temperature_2m")]
    
    leadtime_dailymax = []
    for col in temp_cols:
        leadtime_dailymax.append({
            "city": "Hong Kong",
            "date": event_date,
            "forecast_variable": col,
            "daily_max_from_hourly": pd.to_numeric(hk_prev_hourly_df[col], errors="coerce").max(),
            "unit": "celsius",
            "source": "Open-Meteo Previous Runs API",
        })
    
    leadtime_dailymax_df = pd.DataFrame(leadtime_dailymax)
    display(hk_prev_hourly_df.head())
    display(leadtime_dailymax_df)
else:
    leadtime_dailymax_df = pd.DataFrame()
    print("No previous-runs hourly data available.")

,time,temperature_2m,temperature_2m_previous_day1,temperature_2m_previous_day2,temperature_2m_previous_day3,temperature_2m_previous_day4,temperature_2m_previous_day5,temperature_2m_previous_day6,temperature_2m_previous_day7
0,2026-06-10 00:00:00,23.1,23.0,23.3,22.6,23.0,23.9,24.6,24.8
1,2026-06-10 01:00:00,22.9,22.7,23.2,22.5,23.0,23.5,24.6,24.6
2,2026-06-10 02:00:00,23.5,22.1,22.9,23.1,23.1,22.7,24.6,24.5
3,2026-06-10 03:00:00,23.2,22.0,22.9,23.2,23.0,22.6,24.6,24.4
4,2026-06-10 04:00:00,23.0,21.9,22.9,23.1,23.0,22.6,24.7,24.4


,city,date,forecast_variable,daily_max_from_hourly,unit,source
0,Hong Kong,2026-06-10,temperature_2m,27.4,celsius,Open-Meteo Previous Runs API
1,Hong Kong,2026-06-10,temperature_2m_previous_day1,28.2,celsius,Open-Meteo Previous Runs API
2,Hong Kong,2026-06-10,temperature_2m_previous_day2,30.0,celsius,Open-Meteo Previous Runs API
3,Hong Kong,2026-06-10,temperature_2m_previous_day3,29.8,celsius,Open-Meteo Previous Runs API
4,Hong Kong,2026-06-10,temperature_2m_previous_day4,27.9,celsius,Open-Meteo Previous Runs API
5,Hong Kong,2026-06-10,temperature_2m_previous_day5,28.5,celsius,Open-Meteo Previous Runs API
6,Hong Kong,2026-06-10,temperature_2m_previous_day6,26.0,celsius,Open-Meteo Previous Runs API
7,Hong Kong,2026-06-10,temperature_2m_previous_day7,26.6,celsius,Open-Meteo Previous Runs API


## 6. Compare realised/proxy-realised daily max with lead-time forecast daily max

This is the first rough structure for a future post-processing dataset:

`event date + city + realised temperature + forecast lead-time temperature`

In [16]:
def extract_lead_time_days(variable_name):
    if variable_name == "temperature_2m":
        return 0
    match = re.search(r"previous_day(\d+)", variable_name)
    return int(match.group(1)) if match else None

if len(leadtime_dailymax_df) > 0 and len(archive_daily_df) > 0:
    hk_realised_proxy = archive_daily_df[
        archive_daily_df["city"] == "Hong Kong"
    ]["temperature_2m_max"].iloc[0]
    
    comparison_df = leadtime_dailymax_df.copy()
    comparison_df["lead_time_days_before_valid_time"] = comparison_df[
        "forecast_variable"
    ].apply(extract_lead_time_days)
    
    comparison_df["realised_proxy_temperature_2m_max"] = hk_realised_proxy
    comparison_df["forecast_error"] = (
        comparison_df["daily_max_from_hourly"]
        - comparison_df["realised_proxy_temperature_2m_max"]
    )
    
    comparison_df = comparison_df[[
        "city",
        "date",
        "lead_time_days_before_valid_time",
        "forecast_variable",
        "daily_max_from_hourly",
        "realised_proxy_temperature_2m_max",
        "forecast_error",
        "unit",
        "source",
    ]].sort_values("lead_time_days_before_valid_time")
    
    display(comparison_df)
else:
    comparison_df = pd.DataFrame()
    print("Comparison table not available.")

,city,date,lead_time_days_before_valid_time,forecast_variable,daily_max_from_hourly,realised_proxy_temperature_2m_max,forecast_error,unit,source
0,Hong Kong,2026-06-10,0,temperature_2m,27.4,27.4,0.0,celsius,Open-Meteo Previous Runs API
1,Hong Kong,2026-06-10,1,temperature_2m_previous_day1,28.2,27.4,0.8,celsius,Open-Meteo Previous Runs API
2,Hong Kong,2026-06-10,2,temperature_2m_previous_day2,30.0,27.4,2.6,celsius,Open-Meteo Previous Runs API
3,Hong Kong,2026-06-10,3,temperature_2m_previous_day3,29.8,27.4,2.4,celsius,Open-Meteo Previous Runs API
4,Hong Kong,2026-06-10,4,temperature_2m_previous_day4,27.9,27.4,0.5,celsius,Open-Meteo Previous Runs API
5,Hong Kong,2026-06-10,5,temperature_2m_previous_day5,28.5,27.4,1.1,celsius,Open-Meteo Previous Runs API
6,Hong Kong,2026-06-10,6,temperature_2m_previous_day6,26.0,27.4,-1.4,celsius,Open-Meteo Previous Runs API
7,Hong Kong,2026-06-10,7,temperature_2m_previous_day7,26.6,27.4,-0.8,celsius,Open-Meteo Previous Runs API


## Key numerical result from the Hong Kong sample

For the sample Hong Kong event date, the Open-Meteo proxy-realised daily maximum is 27.4°C.

The Previous Runs API gives lead-time-specific daily maximum forecasts. In this sample, the forecast daily maximum ranges from 26.0°C to 30.0°C across day-0 to day-7 lead times. The forecast error relative to the Open-Meteo realised proxy ranges from -1.4°C to +2.6°C.

This confirms that the project can construct a basic forecast-error dataset of the form:

`city + event date + lead time + forecast daily max + realised/proxy realised daily max + forecast error`

The next methodological step is to convert each lead-time forecast into a predictive distribution, then map that distribution into Polymarket temperature-bin probabilities.

## 7. Optional: Historical Forecast API check

This endpoint gives a continuous historical forecast-style time series. It is not the same as selecting one model run, but it may be useful for bias-correction baselines.

In [19]:
try:
    hk_hist_forecast = get_json(
        HIST_FORECAST_BASE,
        params={
            "latitude": hk["latitude"],
            "longitude": hk["longitude"],
            "start_date": event_date,
            "end_date": event_date,
            "hourly": "temperature_2m",
            "daily": "temperature_2m_max",
            "timezone": hk["timezone"],
            "temperature_unit": "celsius",
        }
    )
    pretty(hk_hist_forecast, max_chars=2000)
except Exception as e:
    hk_hist_forecast = None
    print("Historical Forecast API failed:", e)

URL: https://historical-forecast-api.open-meteo.com/v1/forecast?latitude=22.3022&longitude=114.1746&start_date=2026-06-10&end_date=2026-06-10&hourly=temperature_2m&daily=temperature_2m_max&timezone=Asia%2FHong_Kong&temperature_unit=celsius
Status code: 200
{
  "latitude": 22.319859,
  "longitude": 114.198555,
  "generationtime_ms": 0.42438507080078125,
  "utc_offset_seconds": 28800,
  "timezone": "Asia/Hong_Kong",
  "timezone_abbreviation": "GMT+8",
  "elevation": 38.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "°C"
  },
  "hourly": {
    "time": [
      "2026-06-10T00:00",
      "2026-06-10T01:00",
      "2026-06-10T02:00",
      "2026-06-10T03:00",
      "2026-06-10T04:00",
      "2026-06-10T05:00",
      "2026-06-10T06:00",
      "2026-06-10T07:00",
      "2026-06-10T08:00",
      "2026-06-10T09:00",
      "2026-06-10T10:00",
      "2026-06-10T11:00",
      "2026-06-10T12:00",
      "2026-06-10T13:00",
      "2026-06-10T14:00",
      "2026-06-10T15:00",
      

In [20]:
if hk_hist_forecast is not None and "daily" in hk_hist_forecast:
    hist_forecast_daily_df = pd.DataFrame({
        "city": ["Hong Kong"],
        "date": hk_hist_forecast["daily"]["time"],
        "temperature_2m_max": hk_hist_forecast["daily"]["temperature_2m_max"],
        "unit": hk_hist_forecast.get("daily_units", {}).get("temperature_2m_max"),
        "source": "Open-Meteo Historical Forecast API",
    })
    display(hist_forecast_daily_df)
else:
    hist_forecast_daily_df = pd.DataFrame()
    print("No historical forecast daily data available.")

,city,date,temperature_2m_max,unit,source
0,Hong Kong,2026-06-10,27.4,°C,Open-Meteo Historical Forecast API


## 8. Save lightweight local outputs

These files are saved locally for inspection but are ignored by git.

In [22]:
os.makedirs("../data/raw/weather", exist_ok=True)
os.makedirs("../data/processed/weather", exist_ok=True)

source_audit.to_csv("../data/processed/weather/weather_source_audit.csv", index=False)
archive_daily_df.to_csv("../data/processed/weather/openmeteo_archive_daily_sample.csv", index=False)

if len(hk_hourly_df) > 0:
    hk_hourly_df.to_csv("../data/processed/weather/openmeteo_hk_hourly_sample.csv", index=False)

if len(leadtime_dailymax_df) > 0:
    leadtime_dailymax_df.to_csv("../data/processed/weather/openmeteo_previous_runs_dailymax_sample.csv", index=False)

if len(comparison_df) > 0:
    comparison_df.to_csv("../data/processed/weather/openmeteo_forecast_vs_realised_proxy_sample.csv", index=False)

if hk_archive is not None:
    with open("../data/raw/weather/openmeteo_hk_archive_response.json", "w") as f:
        json.dump(hk_archive, f, indent=2)

if hk_prev_runs is not None:
    with open("../data/raw/weather/openmeteo_hk_previous_runs_response.json", "w") as f:
        json.dump(hk_prev_runs, f, indent=2)

if len(hist_forecast_daily_df) > 0:
    hist_forecast_daily_df.to_csv("../data/processed/weather/openmeteo_historical_forecast_daily_sample.csv", index=False)

print("Saved local weather exploration outputs. These files are ignored by git.")

Saved local weather exploration outputs. These files are ignored by git.


## Settlement-source matching caveat

The Open-Meteo data retrieved in this notebook should be treated as a weather-data feasibility proxy, not as the final settlement truth.

For Hong Kong, the Polymarket contract refers to the Hong Kong Observatory Daily Extract and the “Absolute Daily Max (deg. C)”. For London and New York examples, the relevant settlement sources are specific Wunderground airport stations. Therefore, the final dissertation pipeline should distinguish between:

1. forecast or reanalysis proxy data, such as Open-Meteo;
2. exact realised settlement data, such as Hong Kong Observatory or Wunderground station data;
3. model-implied probabilities derived from forecast distributions;
4. market-implied probabilities derived from Polymarket YES prices.

The key data-quality task is to avoid comparing Polymarket prices against a realised temperature definition that does not match the contract rule.

## Current Priority 6 result

This notebook tests weather-data availability for the Polymarket weather-trading dissertation.

Current findings:

- Open-Meteo Historical Weather API successfully returns daily and hourly 2m temperature data for the candidate cities.
- For Hong Kong on the sample event date, the hourly temperature series reconstructs the daily maximum temperature, confirming that daily maxima can be derived from hourly data.
- Open-Meteo Previous Runs API successfully returns lead-time-specific forecast variables. This provides a first structure for studying how forecast information changes before settlement.
- A forecast-versus-realised-proxy table was created, with city, event date, forecast lead time, forecast daily maximum, realised/proxy realised daily maximum and forecast error.
- Local weather exploration outputs were saved under the ignored data folder.

Main caveat:

Open-Meteo is currently used as a convenient forecast/reanalysis proxy. It is not yet the exact settlement source for the Polymarket contracts. The next data task is to retrieve exact realised data from the Hong Kong Observatory for Hong Kong contracts and from the relevant Wunderground station pages for London and New York contracts where possible.

For the dissertation, the preliminary weather-data structure is:

`city + event date + forecast lead time + forecast daily max temperature + realised/proxy realised daily max temperature`

This can later be joined with the Polymarket YES-price distribution from Priority 5.